# PySpark End-to-End Example: Food Delivery Analytics

**Runs directly in Google Colab — no file uploads needed.** All sample data is created inline in the notebook.

## Scenario

You're a data analyst at **QuickBite**, a food delivery app operating in several cities. The operations team wants answers from last month's delivery data:

1. What's our total revenue, and how does it break down by cuisine?
2. Which restaurants have the best (and worst) average delivery times?
3. Which restaurants are top-rated within each city?
4. How many deliveries were "late" (over 45 minutes), and which restaurants cause the most of them?
5. Who are our most frequent customers?

Same pipeline shape every time: **create a Spark session → load data → clean/transform → join → analyze → save results.**


## Step 0 — Install PySpark

Colab already has Java installed, so this is the only setup step.


In [ ]:
!pip install pyspark -q
print("PySpark installed.")


## Step 1 — Create a SparkSession


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("QuickBite_Delivery_Analytics")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)
spark


## Step 2 — The Data

Two small tables, created directly in Python:

- **orders**: one row per delivery (order id, customer, restaurant, food item, price, order time, delivery time in minutes, customer rating)
- **restaurants**: a lookup table (restaurant id, name, cuisine, city)


In [ ]:
orders_data = [
    (5001, "U01", "R01", "Margherita Pizza",   14.50, "2024-04-01 12:15", 38, 5),
    (5002, "U02", "R02", "Chicken Ramen",      12.00, "2024-04-01 13:02", 52, 3),
    (5003, "U03", "R01", "Pepperoni Pizza",    16.00, "2024-04-02 18:40", 41, 4),
    (5004, "U01", "R03", "Beef Tacos",         10.50, "2024-04-02 19:05", 29, 5),
    (5005, "U04", "R02", "Veggie Ramen",       11.00, "2024-04-03 12:30", 47, 4),
    (5006, "U05", "R04", "California Roll",    18.00, "2024-04-03 20:10", 55, 2),
    (5007, "U02", "R01", "Margherita Pizza",   14.50, "2024-04-04 12:50", 33, 5),
    (5008, "U06", "R03", "Chicken Burrito",    11.50, "2024-04-04 19:20", 60, 2),
    (5009, "U01", "R02", "Chicken Ramen",      12.00, "2024-04-05 13:15", 44, 4),
    (5010, "U03", "R04", "Salmon Sushi Set",   22.00, "2024-04-05 20:00", 39, 5),
    (5011, "U07", "R01", "Pepperoni Pizza",    16.00, "2024-04-06 12:05", 36, 4),
    (5012, "U04", "R03", "Beef Tacos",         10.50, "2024-04-06 18:45", 31, 5),
    (5013, "U08", "R04", "California Roll",    18.00, "2024-04-07 19:30", 49, 3),
    (5014, "U02", "R02", "Veggie Ramen",       11.00, "2024-04-07 12:10", 58, 2),
    (5015, "U05", "R01", "Margherita Pizza",   14.50, "2024-04-08 13:00", 34, 5),
    (5016, "U06", "R04", "Salmon Sushi Set",   22.00, "2024-04-08 20:15", 43, 4),
    (5017, "U01", "R03", "Chicken Burrito",    11.50, "2024-04-09 19:00", 27, 5),
    (5018, "U07", "R02", "Chicken Ramen",      12.00, "2024-04-09 12:40", 63, 1),
    (5019, "U03", "R01", "Pepperoni Pizza",    16.00, "2024-04-10 18:20", 40, 4),
    (5020, "U08", "R04", "California Roll",    18.00, "2024-04-10 20:05", 35, 5),
]
orders_columns = [
    "order_id", "customer_id", "restaurant_id", "food_item",
    "price", "order_time", "delivery_minutes", "rating"
]

restaurants_data = [
    ("R01", "Bella Napoli",    "Italian",  "Springfield"),
    ("R02", "Ramen House",     "Japanese", "Shelbyville"),
    ("R03", "Taco Fiesta",     "Mexican",  "Springfield"),
    ("R04", "Sakura Sushi",    "Japanese", "Shelbyville"),
]
restaurants_columns = ["restaurant_id", "restaurant_name", "cuisine", "city"]

orders = spark.createDataFrame(orders_data, orders_columns)
orders = orders.withColumn("order_time", F.to_timestamp("order_time", "yyyy-MM-dd HH:mm"))

restaurants = spark.createDataFrame(restaurants_data, restaurants_columns)

print("Orders:")
orders.show(5)
print("Restaurants:")
restaurants.show()


## Step 3 — Clean & Transform

Check for data issues, then add a `delivery_status` column classifying each order as `On Time` (<= 45 min) or `Late` (> 45 min).


In [ ]:
issues = orders.filter(
    F.col("order_id").isNull() |
    F.col("customer_id").isNull() |
    (F.col("price") <= 0) |
    (F.col("delivery_minutes") <= 0)
)
print("Rows with data quality issues:", issues.count())

orders = orders.withColumn(
    "delivery_status",
    F.when(F.col("delivery_minutes") <= 45, "On Time").otherwise("Late")
)
orders = orders.withColumn("order_date", F.to_date("order_time"))

orders.select("order_id", "food_item", "delivery_minutes", "delivery_status", "rating").show(5)


## Step 4 — Join Orders With Restaurant Info


In [ ]:
deliveries = orders.join(restaurants, on="restaurant_id", how="left")
deliveries.select("order_id", "restaurant_name", "cuisine", "city", "price", "delivery_status", "rating").show(5)


## Step 5 — Answer the Business Questions

### Q1: Total revenue, and revenue by cuisine


In [ ]:
total_revenue = deliveries.agg(F.round(F.sum("price"), 2).alias("total_revenue")).collect()[0]["total_revenue"]
print(f"Total revenue: ${total_revenue:,.2f}")

revenue_by_cuisine = (
    deliveries.groupBy("cuisine")
    .agg(
        F.round(F.sum("price"), 2).alias("revenue"),
        F.count("order_id").alias("num_orders"),
    )
    .orderBy(F.col("revenue").desc())
)
revenue_by_cuisine.show()


### Q2: Average delivery time by restaurant


In [ ]:
avg_delivery_by_restaurant = (
    deliveries.groupBy("restaurant_name", "city")
    .agg(
        F.round(F.avg("delivery_minutes"), 1).alias("avg_delivery_minutes"),
        F.count("order_id").alias("num_orders"),
    )
    .orderBy(F.col("avg_delivery_minutes").desc())
)
avg_delivery_by_restaurant.show()


### Q3: Top-rated restaurant within each city (window function)

We rank restaurants by average rating **within each city**, without collapsing the row-level detail — a classic use of `Window.partitionBy()`.


In [ ]:
restaurant_ratings = (
    deliveries.groupBy("restaurant_name", "city")
    .agg(F.round(F.avg("rating"), 2).alias("avg_rating"))
)

city_window = Window.partitionBy("city").orderBy(F.col("avg_rating").desc())
ranked_restaurants = restaurant_ratings.withColumn("rank_in_city", F.row_number().over(city_window))

ranked_restaurants.orderBy("city", "rank_in_city").show()

print("Top restaurant per city:")
ranked_restaurants.filter(F.col("rank_in_city") == 1).show()


### Q4: Late deliveries — how many, and which restaurants cause the most?


In [ ]:
late_count = deliveries.filter(F.col("delivery_status") == "Late").count()
total_count = deliveries.count()
print(f"Late deliveries: {late_count} out of {total_count} ({late_count / total_count:.0%})")

late_by_restaurant = (
    deliveries.filter(F.col("delivery_status") == "Late")
    .groupBy("restaurant_name")
    .agg(F.count("order_id").alias("num_late_orders"))
    .orderBy(F.col("num_late_orders").desc())
)
late_by_restaurant.show()


### Q5: Most frequent customers


In [ ]:
top_customers = (
    deliveries.groupBy("customer_id")
    .agg(
        F.count("order_id").alias("num_orders"),
        F.round(F.sum("price"), 2).alias("total_spend"),
    )
    .orderBy(F.col("num_orders").desc(), F.col("total_spend").desc())
    .limit(5)
)
top_customers.show()


## Step 6 — Save the Results

Write the cuisine revenue summary out to Parquet — the standard analytics-ready format (Colab's local disk, under `/content/`).


In [ ]:
revenue_by_cuisine.write.mode("overwrite").parquet("/content/revenue_by_cuisine_parquet")
print("Saved to /content/revenue_by_cuisine_parquet")
print("(Use the Files panel on the left in Colab to browse and download it.)")


## Step 7 — Same Analysis in Plain SQL


In [ ]:
deliveries.createOrReplaceTempView("deliveries")

spark.sql("""
    SELECT cuisine,
           ROUND(SUM(price), 2) AS revenue,
           COUNT(order_id) AS num_orders,
           ROUND(AVG(delivery_minutes), 1) AS avg_delivery_minutes
    FROM deliveries
    GROUP BY cuisine
    ORDER BY revenue DESC
""").show()


## Summary

In this notebook you:
- Created inline sample DataFrames for `orders` and `restaurants` (no upload required)
- Cleaned data and derived a `delivery_status` column with `when`/`otherwise`
- Joined orders to restaurant details
- Used `groupBy().agg()` for revenue and delivery-time analysis
- Used a **window function** (`row_number()` partitioned by city) to rank restaurants within each city
- Filtered for a business condition (late deliveries) and aggregated by restaurant
- Wrote results to Parquet
- Reproduced the same analysis in plain Spark SQL

Swap the inline `orders_data`/`restaurants_data` lists for `spark.read.csv(...)` (or a database/data lake read) and this exact pipeline scales from a few rows on Colab to millions of deliveries on a real cluster.


In [ ]:
spark.stop()
print("Spark session stopped.")
